# 00 · Extracción — IVR Alkosto (datos junio 2026)

Reemplaza al CSV histórico de Interaxa (`vw_interaxa_detalle_ivr_...csv`) que usaba
`01_Alk_final.ipynb`. Consulta directamente `vw_emt_gc_detalle_ivr` y aplana
`atributos_custom` en columnas `customN`.

**Salida:** `data/00_raw/df_raw_<fecha_inicio>_<fecha_fin>.parquet` — insumo del
Notebook 2 (`02_reconstruccion_traza.ipynb`).

**Requisitos:**
- Archivo `.env` en la raíz del proyecto (mismo nivel que este notebook o un nivel
  arriba) con las credenciales de conexión.
- `pip install python-dotenv psycopg2-binary pandas pyarrow` (pyarrow para exportar
  a parquet; si prefieres solo Excel, puedes omitirlo y usar `.xlsx` al final).

In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
import psycopg2
from dotenv import load_dotenv

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

### Parámetros del análisis

Ajusta aquí el periodo y la división antes de correr el resto del notebook. Todo lo
que sigue depende de estos valores — no hay rutas ni fechas quemadas más abajo.

In [2]:
# --- Parámetros editables ---
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DIVISIONES = ["Alkosto", "Home,Alkosto", "Alkosto,Home"]

ORGANIZACION = "emtelcosas"

# Carpeta raíz de datos del proyecto (ajusta si tu estructura es distinta)
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
print(f"Salida esperada: {OUTPUT_PATH}")

Salida esperada: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet


### Conexión

In [3]:
load_dotenv()  # busca un archivo .env en el directorio actual o en los padres

DB_USER = os.getenv("IVR_DB_USER")
DB_PASSWORD = os.getenv("IVR_DB_PASSWORD")
DB_HOST = os.getenv("IVR_DB_HOST")
DB_PORT = int(os.getenv("IVR_DB_PORT", "5432"))
DB_NAME = os.getenv("IVR_DB_NAME")

faltantes = [
    nombre
    for nombre, valor in {
        "IVR_DB_USER": DB_USER,
        "IVR_DB_PASSWORD": DB_PASSWORD,
        "IVR_DB_HOST": DB_HOST,
        "IVR_DB_NAME": DB_NAME,
    }.items()
    if not valor
]
if faltantes:
    raise RuntimeError(
        f"Faltan variables de entorno en tu .env: {faltantes}. "
        "Revisa .env para ver los nombres esperados."
    )

In [6]:
con = psycopg2.connect(
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
)

### Consulta parametrizada

Usa placeholders `%s` (psycopg2 los sustituye de forma segura) en vez de f-strings,
para no repetir el patrón de fechas/organización.

In [7]:
sql_query = """
SELECT *
FROM public.vw_emt_gc_detalle_ivr
WHERE organizacion = %(organizacion)s
  AND division = ANY(%(divisiones)s)
  AND fecha_inicio >= %(fecha_inicio)s
  AND fecha_inicio <= %(fecha_fin)s
ORDER BY fecha_inicio;
"""

params = {
    "organizacion": ORGANIZACION,
    "divisiones": DIVISIONES,
    "fecha_inicio": FECHA_INICIO,
    "fecha_fin": FECHA_FIN,
}

df = pd.read_sql_query(sql_query, con, params=params)
con.close()
print(df.shape)
df.head(3)

(105499, 23)


,organizacion,id_conversacion,ani,dnis,division,nombre_ivr,fecha_hora_ingreso,fecha_hora_fin,duracion_ivr,duracion_navegacion,duracion_transaccional,duracion_paso_ce,duracion_desborde,traza_opciones,ultima_opcion,tipo_ult_opcion,paso_ce,tipo_desconexion,id_campana,atributos_custom,custom_50,fecha_inicio,fecha_fin
0,emtelcosas,891f883b-e8c0-4a43-91a8-00f93459a745,+573212202672,+576014073033,Alkosto,"ALK_IVR_PRINCIPAL,Default In-Queue Flow Alkost...",2026-06-01 18:51:32.530,2026-06-01 19:03:54.720,185663,185095.0,1496.0,284.0,0.0,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,1003;Finalización por fin de flujo;31,None,SI,System,None,{'custom2': '|2;Numero documento ingresado;102...,None,2026-06-01,2026-06-01
1,emtelcosas,a474401c-9cf1-491b-a677-0ac154ac6141,+573156554180,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 15:31:24.227,2026-06-01 15:32:22.083,57833,57301.0,1301.0,266.0,0.0,|0;Inicio IVR ;0|999;No input;14897,999;No input;14897,None,NO,External,None,{'custom2': '|2;Numero documento ingresado;100...,None,2026-06-01,2026-06-01
2,emtelcosas,352a5eb7-1bfd-484f-820e-12503f2f8c56,+573146826588,+576014073033,Alkosto,ALK_IVR_PRINCIPAL,2026-06-01 12:23:54.010,2026-06-01 12:34:28.390,36453,34771.0,1682.0,NaN,0.0,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,9;Transferencia Alkosto - Tuya;14753,None,NO,System,None,{'custom16': '|12;Usuario_Identificado_Con_ANI...,None,2026-06-01,2026-06-01


### Chequeos rápidos de sanidad

Antes de aplanar `atributos_custom`, valida que la extracción trajo lo esperado:
que no venga vacía, que las fechas caigan dentro del rango pedido y que
`id_conversacion` no tenga duplicados exactos.

In [8]:
assert len(df) > 0, "La consulta no trajo filas — revisa parámetros de fecha/división."

print("Filas:", len(df))
print("id_conversacion únicos:", df["id_conversacion"].nunique())
print("Duplicados exactos de id_conversacion:", df["id_conversacion"].duplicated().sum())
print("Rango fecha_inicio:", df["fecha_inicio"].min(), "→", df["fecha_inicio"].max())
print("Divisiones encontradas:", df["division"].unique())
print("\nNulos en traza_opciones:", df["traza_opciones"].isna().sum())

Filas: 105499
id_conversacion únicos: 105499
Duplicados exactos de id_conversacion: 0
Rango fecha_inicio: 2026-06-01 → 2026-06-30
Divisiones encontradas: <ArrowStringArray>
['Alkosto']
Length: 1, dtype: str

Nulos en traza_opciones: 8348


### Aplanado de `atributos_custom`

Cada fila trae un dict con claves `customN`, cada una con un único paso `codigo;texto;tiempo`. Los separamos en
columnas propias para poder reconstruir la traza completa en el Notebook 2.

In [12]:
df_flat = df.join(df["atributos_custom"].apply(pd.Series))

columnas_custom = [c for c in df_flat.columns if c.startswith("custom")]
print(df_flat.shape)
print(f"Columnas custom encontradas ({len(columnas_custom)}):", sorted(columnas_custom))
df_flat[["id_conversacion", "traza_opciones"] + columnas_custom].head(3)

(105499, 42)
Columnas custom encontradas (20): ['custom11', 'custom15', 'custom16', 'custom2', 'custom21', 'custom22', 'custom24', 'custom25', 'custom26', 'custom28', 'custom29', 'custom3', 'custom30', 'custom31', 'custom32', 'custom33', 'custom34', 'custom47', 'custom48', 'custom_50']


,id_conversacion,traza_opciones,custom_50,custom2,custom16,custom21,custom22,custom28,custom29,custom30,custom3,custom11,custom15,custom24,custom25,custom32,custom34,custom47,custom31,custom33,custom26,custom48
0,891f883b-e8c0-4a43-91a8-00f93459a745,|0;Inicio IVR ;0|3;Habeas data positivo;48030|...,None,|2;Numero documento ingresado;1023941473,|12;Usuario_Identificado_Con_ANI;NO,|101;Consulta ws ActualizaHabeasData;FAILURE,|100;Consulta ws ConsultaHabeasData;FAILURE,|107;Consulta ws CheckAftersalesCases;OK,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,a474401c-9cf1-491b-a677-0ac154ac6141,|0;Inicio IVR ;0|999;No input;14897,None,|2;Numero documento ingresado;1007469531,|12;Usuario_Identificado_Con_ANI;NO,NaN,|100;Consulta ws ConsultaHabeasData;FAILURE,NaN,|108;Consulta ws GetClientByAni;NOK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,352a5eb7-1bfd-484f-820e-12503f2f8c56,|0;Inicio IVR ;0|20;Usuario_Identificado_Con_A...,None,NaN,|12;Usuario_Identificado_Con_ANI;SI,NaN,NaN,NaN,|108;Consulta ws GetClientByAni;OK,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Exportar

Se guarda en `data/00_raw/` con el rango de fechas en el nombre, para poder tener
varias extracciones (distintos periodos) sin pisarlas entre sí. `parquet`
preserva mejor los tipos (fechas, dict) que Excel; si necesitas revisarlo a ojo,
exporta también una copia en `.xlsx` de una muestra pequeña.

In [14]:
df_flat.to_parquet(OUTPUT_PATH, index=False)
print(f"Guardado: {OUTPUT_PATH}  ({len(df_flat)} filas)")

# Muestra legible en Excel para revisión manual rápida (primeras 200 filas)
muestra_path = RAW_DIR / f"df_raw_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_flat.head(200).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet  (105499 filas)
Muestra Excel: data\00_raw\df_raw_muestra_2026-06-01_2026-06-30.xlsx


# 01 · Reconstrucción de traza — IVR Alkosto

En el pipeline original, todos los pasos de una interacción (navegación,
validaciones de flujo, llamadas a webservice, paso a asesor) llegaban mezclados
en una sola columna (`opcionesnavegaciontrazaopciones`). En los datos actuales,
la vista los separa: `traza_opciones` trae la navegación principal y cada
columna `customN` trae **un solo paso** adicional (validaciones, resultados de
webservice, etc.).

Confirmamos contra el PDF del flujo (`Flujo_actualizado_Alkosto_corte_28-04-26`)
que esos pasos de `customN` **sí son nodos de decisión reales** del árbol (carriles
"API": `¿Error de consulta?`, `¿Estado Fraude?`, `¿Tiene PEA?`, etc.), así que se
integran todos — no se excluyen.

**Entrada:** parquet generado por `00_extraccion.ipynb`.

**Salida:**
- `df_pasos_<periodo>.parquet` — formato largo (un paso por fila), ya con el orden
  cronológico correcto. Este es el insumo directo del Notebook 3 (reemplaza el
  `split('|')` + `stack()` que hacía `01_Alk_final.ipynb` sobre la columna cruda).
- `df_traza_completa_<periodo>.parquet` — un renglón por `id_conversacion` con la
  traza reconstruida como string `|codigo;texto;tiempo|...`, solo para auditoría /
  comparación visual contra `traza_opciones` original.

In [ ]:
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
tqdm.pandas()

### Parámetros — deben coincidir con los usados en `00_extraccion.ipynb`

In [ ]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
RAW_DIR = DATA_DIR / "00_raw"
STAGE_DIR = DATA_DIR / "01_staging"
STAGE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / f"df_raw_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_TRAZA_PATH = STAGE_DIR / f"df_traza_completa_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert INPUT_PATH.exists(), f"No encuentro {INPUT_PATH} — corre primero 01_extraccion.ipynb"
print(f"Leyendo: {INPUT_PATH}")

Leyendo: data\00_raw\df_raw_2026-06-01_2026-06-30.parquet


In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)

ID_COL = "id_conversacion"
COL_TRAZA_PRINCIPAL = "traza_opciones"
columnas_custom = sorted(
    [c for c in df.columns if c.startswith("custom") and c not in ("custom_50",)],
    key=lambda c: int(c.replace("custom", "")) if c.replace("custom", "").isdigit() else 999,
)
print("Columnas fuente que se van a integrar:", [COL_TRAZA_PRINCIPAL] + columnas_custom)

(105499, 42)
Columnas fuente que se van a integrar: ['traza_opciones', 'custom2', 'custom3', 'custom11', 'custom15', 'custom16', 'custom21', 'custom22', 'custom24', 'custom25', 'custom26', 'custom28', 'custom29', 'custom30', 'custom31', 'custom32', 'custom33', 'custom34', 'custom47', 'custom48']


### Parseo de un campo tipo `|codigo;texto;tiempo|codigo;texto;tiempo...`

Tanto `traza_opciones` (varios pasos) como cada `customN` (un solo paso) usan el
mismo formato interno, así que se parsean con la misma función.

In [ ]:
def parsear_pasos(valor):
    """Convierte '|cod;texto;tiempo|cod;texto;tiempo' en una lista de tuplas
    (codigo, texto, tiempo). Ignora vacíos y fragmentos mal formados (no 3 partes).
    Devuelve también el conteo de fragmentos descartados por mal formados.
    """
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return [], 0
    partes = str(valor).split("|")
    pasos = []
    descartados = 0
    for parte in partes:
        parte = parte.strip()
        if not parte:
            continue
        campos = parte.split(";")
        if len(campos) != 3:
            descartados += 1
            continue
        codigo, texto, tiempo = (c.strip() for c in campos)
        pasos.append((codigo, texto, tiempo))
    return pasos, descartados

### Reconstrucción por `id_conversacion`

Por cada fila: se juntan los pasos de `traza_opciones` + todas las `customN` no
nulas, etiquetando de dónde vino cada paso (`origen_campo`), y se ordenan por el
código numérico inicial para recuperar la secuencia cronológica real.

In [ ]:
registros_pasos = []  # una fila por paso -> formato largo
registros_traza = []  # una fila por id_conversacion -> string reconstruido, para auditoría
total_descartados = 0
total_codigo_no_numerico = 0

columnas_fuente = [COL_TRAZA_PRINCIPAL] + columnas_custom

for row in tqdm(df[[ID_COL] + columnas_fuente].itertuples(index=False), total=len(df)):
    id_conv = getattr(row, ID_COL)
    pasos_fila = []  # (codigo_str, codigo_num, texto, tiempo, origen_campo)

    for campo in columnas_fuente:
        valor = getattr(row, campo)
        pasos, descartados = parsear_pasos(valor)
        total_descartados += descartados
        for codigo, texto, tiempo in pasos:
            codigo_num = pd.to_numeric(codigo, errors="coerce")
            if pd.isna(codigo_num):
                total_codigo_no_numerico += 1
            pasos_fila.append((codigo, codigo_num, texto, tiempo, campo))

    if not pasos_fila:
        continue

    # Orden cronológico: por código numérico; los no-numéricos van al final,
    # conservando el orden relativo en que llegaron (sort estable).
    pasos_fila.sort(key=lambda p: (pd.isna(p[1]), p[1] if not pd.isna(p[1]) else 0))

    for orden, (codigo, codigo_num, texto, tiempo, origen_campo) in enumerate(pasos_fila):
        registros_pasos.append(
            {
                "id_conversacion": id_conv,
                "orden": orden,
                "op_num": codigo,
                "op_text": texto,
                "op_tiempo": tiempo,
                "origen_campo": origen_campo,
            }
        )

    traza_reconstruida = "".join(f"|{c};{t};{tp}" for c, _, t, tp, _ in pasos_fila)
    registros_traza.append(
        {
            "id_conversacion": id_conv,
            "traza_completa": traza_reconstruida,
            "n_pasos": len(pasos_fila),
            "n_pasos_custom": sum(1 for p in pasos_fila if p[4] != COL_TRAZA_PRINCIPAL),
        }
    )

df_pasos = pd.DataFrame(registros_pasos)
df_traza_completa = pd.DataFrame(registros_traza)

print(f"Fragmentos mal formados descartados: {total_descartados}")
print(f"Pasos con código no numérico (van al final del orden): {total_codigo_no_numerico}")
print(f"id_conversacion sin ningún paso parseable: {len(df) - len(registros_traza)}")

100%|██████████| 105499/105499 [00:49<00:00, 2149.89it/s]


Fragmentos mal formados descartados: 0
Pasos con código no numérico (van al final del orden): 0
id_conversacion sin ningún paso parseable: 8187


### Chequeos de sanidad

Antes de pasar esto al Notebook 3, valida que la integración de `customN` sí esté
aportando pasos (si `n_pasos_custom` fuera 0 en todas las filas, algo falló en el
aplanado del Notebook 1) y que el orden por código no esté generando secuencias
absurdas.

In [ ]:
print("Filas en df_pasos (formato largo):", len(df_pasos))
print("id_conversacion cubiertos:", df_pasos["id_conversacion"].nunique(), "/", df[ID_COL].nunique())
print("\nDistribución de origen_campo:")
print(df_pasos["origen_campo"].value_counts())

print("\n% de conversaciones con al menos un paso proveniente de customN:")
print((df_traza_completa["n_pasos_custom"] > 0).mean().round(3))

print("\nDistribución de cantidad de pasos por conversación:")
print(df_traza_completa["n_pasos"].describe())

Filas en df_pasos (formato largo): 1296067
id_conversacion cubiertos: 97312 / 105499

Distribución de origen_campo:
origen_campo
traza_opciones    818010
custom29           82069
custom16           82010
custom28           46034
custom2            45699
custom34           39896
custom11           35366
custom21           31358
custom24           27810
custom3            27411
custom30           15783
custom25           12139
custom15           11694
custom22            8803
custom32            5166
custom47            3638
custom48            1383
custom31             817
custom33             817
custom26             164
Name: count, dtype: int64

% de conversaciones con al menos un paso proveniente de customN:
0.942

Distribución de cantidad de pasos por conversación:
count    97312.000000
mean        13.318676
std          7.501728
min          1.000000
25%          8.000000
50%         12.000000
75%         17.000000
max         60.000000
Name: n_pasos, dtype: float64


In [ ]:
# Inspección manual de un caso con pasos custom, para comparar visualmente
# contra la traza_opciones original y confirmar que el orden quedó coherente.
ejemplo_id = df_traza_completa.loc[df_traza_completa["n_pasos_custom"] > 0, "id_conversacion"].iloc[0]
print("Ejemplo:", ejemplo_id)
print("\ntraza_opciones original:")
print(df.loc[df[ID_COL] == ejemplo_id, COL_TRAZA_PRINCIPAL].values[0])
print("\nTraza reconstruida (orden final):")
df_pasos[df_pasos["id_conversacion"] == ejemplo_id].sort_values("orden")

Ejemplo: 891f883b-e8c0-4a43-91a8-00f93459a745

traza_opciones original:
|0;Inicio IVR ;0|3;Habeas data positivo;48030|17;Menu principal;346|7;Garantias y devoluciones ;24254|14;Repetir informacion;26061|997;Repeat;23|523;Iniciar_Tu_Garantia;9781|527;Igual_O_Menor_30_Dias;8972|540;Producto_Deteriorado;5824|528;Gran_Tamano;14839|204;Paso agente garantias;132|900;Bienvenida encuesta SAC;569944|901;Primera pregunta SAC;38|902;Segunda pregunta SAC;16857|906;Pasa a buzon = NO;13125|1003;Finalización por fin de flujo;31

Traza reconstruida (orden final):


,id_conversacion,orden,op_num,op_text,op_tiempo,origen_campo
0,891f883b-e8c0-4a43-91a8-00f93459a745,0,0,Inicio IVR,0,traza_opciones
1,891f883b-e8c0-4a43-91a8-00f93459a745,1,2,Numero documento ingresado,1023941473,custom2
2,891f883b-e8c0-4a43-91a8-00f93459a745,2,3,Habeas data positivo,48030,traza_opciones
3,891f883b-e8c0-4a43-91a8-00f93459a745,3,7,Garantias y devoluciones,24254,traza_opciones
4,891f883b-e8c0-4a43-91a8-00f93459a745,4,12,Usuario_Identificado_Con_ANI,NO,custom16
5,891f883b-e8c0-4a43-91a8-00f93459a745,5,14,Repetir informacion,26061,traza_opciones
6,891f883b-e8c0-4a43-91a8-00f93459a745,6,17,Menu principal,346,traza_opciones
7,891f883b-e8c0-4a43-91a8-00f93459a745,7,100,Consulta ws ConsultaHabeasData,FAILURE,custom22
8,891f883b-e8c0-4a43-91a8-00f93459a745,8,101,Consulta ws ActualizaHabeasData,FAILURE,custom21
9,891f883b-e8c0-4a43-91a8-00f93459a745,9,107,Consulta ws CheckAftersalesCases,OK,custom28


Revisa el ejemplo impreso arriba: los pasos que vinieron de columnas `customN`
deben aparecer intercalados en la posición cronológica correcta (por código), no
todos al final. Si ves algo raro (por ejemplo, un paso de validación apareciendo
antes de `Inicio IVR`), revisa `total_codigo_no_numerico` — puede haber códigos
con formato distinto que necesiten un tratamiento especial antes de ordenar.

### Exportar

In [ ]:
df_pasos.to_parquet(OUT_PASOS_PATH, index=False)
df_traza_completa.to_parquet(OUT_TRAZA_PATH, index=False)
print(f"Guardado: {OUT_PASOS_PATH}  ({len(df_pasos)} filas)")
print(f"Guardado: {OUT_TRAZA_PATH}  ({len(df_traza_completa)} filas)")

# Muestra Excel para revisión manual
muestra_path = STAGE_DIR / f"df_pasos_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_pasos.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\01_staging\df_pasos_2026-06-01_2026-06-30.parquet  (1296067 filas)
Guardado: data\01_staging\df_traza_completa_2026-06-01_2026-06-30.parquet  (97312 filas)
Muestra Excel: data\01_staging\df_pasos_muestra_2026-06-01_2026-06-30.xlsx


# 02 · Limpieza, segmentación y aristas — IVR Alkosto

Toma `df_pasos` (salida de `01_alk_traza.ipynb`) y hace lo que hacía la
segunda mitad de `01_Alk_final.ipynb`: cruzar contra el maestro, asignar Nodo,
clasificar cada paso, y construir la arista origen→destino (`op_text_final`) con
`shift(-1)`.

#### Decisión: cruzar por `Llave`, no solo por `CodigoTraza`

`CodigoTraza` no es único en el maestro (5 códigos se repiten con texto distinto).
`Llave` (`CodigoTraza_TrazaOpcion`) sí es 100% única (142/142), así que cruzamos por
ahí, con un cruce de respaldo por texto normalizado (`unidecode`) para diferencias
de tildes.

#### Hallazgo tras la primera corrida: 65.8% de los pasos no matcheó

Revisamos la lista de `no_identificados` contra el PDF del flujo y encontramos que
**no es un bug del cruce** — son tres cosas distintas mezcladas en `op_text`:

1. **Validaciones de identidad / llamadas a webservice** (`Consulta ws
   GetClientByAni`, `Usuario_Identificado_Con_ANI`, `Existe habeas data`...): pasan
   en casi cualquier llamada, el maestro nunca las modeló como nodos del árbol de
   negocio. Solo estas 8 categorías ya son el 45% de todo lo no identificado.
2. **La encuesta de satisfacción SAC** (`Bienvenida encuesta SAC`, `Primera/Segunda
   pregunta SAC`): confirmado en el PDF (pág. 2-4) que es un sub-flujo enlazado
   (`Alk_SAC_Despachos_in_01 → Encuesta → FIN`) al final de casi cada rama, no
   dibujado nodo por nodo en el árbol principal ni en el maestro.
3. **Renombres genuinos**: navegación real con nombre interno viejo
   (`Igual_O_Menor_30_Dias`, `Iniciar_Tu_Garantia`, `Estado_De_Tu_Garantia`,
   `Inicio IVR inbound transfers`) que sí deberían aparecer en el grafo.

**Decisión tomada con el usuario:** el grafo de navegación (Notebooks 4-5) se
construye **solo** con pasos de negocio (`incluir_en_grafo == True`), excluyendo
validaciones técnicas y encuesta SAC. Todo se conserva en `df_transformado` para
análisis posterior en Power BI si hace falta.

**Entrada:** `df_pasos_<periodo>.parquet` (Notebook 2) +
`Maestro_trazas_alkosto.xlsx`.

**Salida:** `df_transformado_<periodo>.parquet`.

In [2]:
import warnings
from pathlib import Path

import pandas as pd
from unidecode import unidecode

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

#### Parámetros — deben coincidir con los notebooks anteriores

In [4]:
FECHA_INICIO = "2026-06-01"
FECHA_FIN = "2026-06-30"

DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
STAGE_DIR = DATA_DIR / "01_staging"
PROC_DIR = DATA_DIR / "02_procesado"
PROC_DIR.mkdir(parents=True, exist_ok=True)

MAESTRO_PATH = Path("C:/Users/jupasoro/OneDrive - Emtelco/Proyectos/Alkosto/grafos_ivr_2026/Maestro trazas alkosto.xlsx")

IN_PASOS_PATH = STAGE_DIR / f"df_pasos_{FECHA_INICIO}_{FECHA_FIN}.parquet"
OUT_TRANSFORMADO_PATH = PROC_DIR / f"df_transformado_{FECHA_INICIO}_{FECHA_FIN}.parquet"

assert IN_PASOS_PATH.exists(), f"No encuentro {IN_PASOS_PATH} — corre primero 02_reconstruccion_traza.ipynb"
assert MAESTRO_PATH.exists(), f"No encuentro {MAESTRO_PATH} — ajusta MAESTRO_PATH"
print(f"Leyendo pasos: {IN_PASOS_PATH}")
print(f"Leyendo maestro: {MAESTRO_PATH}")

Leyendo pasos: data\01_staging\df_pasos_2026-06-01_2026-06-30.parquet
Leyendo maestro: C:\Users\jupasoro\OneDrive - Emtelco\Proyectos\Alkosto\grafos_ivr_2026\Maestro trazas alkosto.xlsx


In [5]:
df_pasos = pd.read_parquet(IN_PASOS_PATH)
print("df_pasos:", df_pasos.shape)
df_pasos.head(3)

df_pasos: (1296067, 6)


,id_conversacion,orden,op_num,op_text,op_tiempo,origen_campo
0,891f883b-e8c0-4a43-91a8-00f93459a745,0,0,Inicio IVR,0,traza_opciones
1,891f883b-e8c0-4a43-91a8-00f93459a745,1,2,Numero documento ingresado,1023941473,custom2
2,891f883b-e8c0-4a43-91a8-00f93459a745,2,3,Habeas data positivo,48030,traza_opciones


#### Cargar y preparar el maestro

In [6]:
maestro = pd.read_excel(MAESTRO_PATH, sheet_name="Detalle MaestroTrazasOpciones")

columnas_nodo = [f"Nodo {i}" for i in range(1, 11)]
columnas_maestro_utiles = [
    "Llave",
    "CodigoTraza",
    "TrazaOpcion",
    "Efectivo",
    "Clasifica Efectivo",
    "Clasifica Traza",
    "TrazaFallaWebServ",
    "VoiceBot",
    "Clasifica Voicebot",
    "Nombre IVR",
    "CodigoOrden",
] + columnas_nodo

maestro = maestro[columnas_maestro_utiles].copy()
maestro["TrazaOpcion"] = maestro["TrazaOpcion"].astype(str).str.strip()

print("Filas en maestro:", len(maestro))
print("Llave únicas:", maestro["Llave"].nunique(), "/", len(maestro))
assert maestro["Llave"].is_unique, "Llave dejó de ser única — revisa el maestro actualizado"
maestro.head(3)

Filas en maestro: 142
Llave únicas: 142 / 142


,Llave,CodigoTraza,TrazaOpcion,Efectivo,Clasifica Efectivo,Clasifica Traza,TrazaFallaWebServ,VoiceBot,Clasifica Voicebot,Nombre IVR,CodigoOrden,Nodo 1,Nodo 2,Nodo 3,Nodo 4,Nodo 5,Nodo 6,Nodo 7,Nodo 8,Nodo 9,Nodo 10
0,0_Inicio IVR,0,Inicio IVR,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11000000000,Inicio IVR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1_Confirmar documento,1,Confirmar documento,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11100000000,Inicio IVR,Confirmar documento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2_Modificar documento,2,Modificar documento,No,NaN,Navegación,No,No,NaN,IVR General Alkosto,11200000000,Inicio IVR,Modificar documento,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Construir `llave` en `df_pasos` con el mismo formato del maestro (`CodigoTraza_TrazaOpcion`)

In [7]:
df_pasos["op_text"] = df_pasos["op_text"].astype(str).str.strip()
df_pasos["op_num_int"] = pd.to_numeric(df_pasos["op_num"], errors="coerce")

codigo_no_numerico = df_pasos["op_num_int"].isna().sum()
if codigo_no_numerico:
    print(f"Aviso: {codigo_no_numerico} pasos con op_num no numérico — no podrán armar 'llave', quedarán 'no identificado'.")

df_pasos["llave"] = (
    df_pasos["op_num_int"].astype("Int64").astype(str) + "_" + df_pasos["op_text"]
)
df_pasos.loc[df_pasos["op_num_int"].isna(), "llave"] = pd.NA

#### Cruce principal por `llave` (exacto: código + texto)

In [8]:
df_join = df_pasos.merge(
    maestro.add_suffix("_maestro").rename(columns={"Llave_maestro": "llave"}),
    on="llave",
    how="left",
)

matcheo_llave = df_join["CodigoTraza_maestro"].notna().mean()
print(f"% de pasos matcheados por llave exacta: {matcheo_llave:.1%}")

% de pasos matcheados por llave exacta: 33.0%


#### Cruce de respaldo por texto normalizado (sin tildes/mayúsculas)

Para los que no matchearon por `llave` exacta. Definimos aquí también
`aplicar_respaldo_texto`, una función reutilizable — la volvemos a usar más abajo
para el crosswalk manual de renombres, así que evitamos duplicar la lógica.

In [9]:
def normalizar(texto):
    return unidecode(str(texto)).strip().lower()

maestro["texto_normalizado"] = maestro["TrazaOpcion"].apply(normalizar)
lookup_texto = (
    maestro.drop_duplicates(subset="texto_normalizado", keep="first")
    .set_index("texto_normalizado")
)
columnas_maestro_suffix = [c for c in df_join.columns if c.endswith("_maestro")]


def aplicar_respaldo_texto(df, mascara, texto_busqueda_col):
    """Para las filas en `mascara`, busca en `lookup_texto` usando el texto
    normalizado de `texto_busqueda_col` y rellena las columnas *_maestro donde
    sigan vacías. Modifica `df` in place y devuelve la máscara de lo que sí
    encontró algo en esta pasada."""
    texto_norm = df.loc[mascara, texto_busqueda_col].apply(normalizar)
    encontro_algo = pd.Series(False, index=df.index)
    for col in columnas_maestro_suffix:
        col_original = col.replace("_maestro", "")
        if col_original not in lookup_texto.columns:
            continue
        valores = texto_norm.map(lookup_texto[col_original])
        df.loc[mascara, col] = df.loc[mascara, col].fillna(valores)
    encontro_algo.loc[mascara] = texto_norm.map(lambda t: t in lookup_texto.index)
    return encontro_algo


sin_match = df_join["CodigoTraza_maestro"].isna()
aplicar_respaldo_texto(df_join, sin_match, "op_text")

df_join["matched_por"] = "llave"
df_join.loc[sin_match & df_join["CodigoTraza_maestro"].notna(), "matched_por"] = "texto_normalizado"
df_join.loc[df_join["CodigoTraza_maestro"].isna(), "matched_por"] = "sin_match"

print(df_join["matched_por"].value_counts())
print(f"\n% total matcheado (llave + respaldo texto): {(df_join['matched_por'] != 'sin_match').mean():.1%}")

matched_por
sin_match            852966
llave                427521
texto_normalizado     15580
Name: count, dtype: int64

% total matcheado (llave + respaldo texto): 34.2%


#### Diagnóstico de lo no identificado (antes del crosswalk)

Ver el detalle completo de esta corrida arriba en las notas del notebook: 85
textos distintos, 852,966 pasos. El top 30 (por frecuencia) cubre ~88% de ese
volumen; se clasificó así:

- **Validación técnica / webservice** (~45% del volumen no identificado):
  `Consulta ws *`, `Usuario_Identificado_Con_ANI`/`Ani`, `Existe habeas data`,
  `Numero documento ingresado`, `Numero documento validado para factura`,
  `Número de factura`, `No input`, `Repeat`, `Pasa a buzon = NO`.
- **Encuesta SAC** (confirmada como sub-flujo aparte en el PDF):
  `Bienvenida encuesta SAC`, `Primera pregunta SAC`, `Segunda pregunta SAC`.
- **Renombres genuinos** (navegación real, nombre interno distinto al maestro):
  `Igual_O_Menor_30_Dias`, `Iniciar_Tu_Garantia`, `Estado_De_Tu_Garantia`,
  `Inicio IVR inbound transfers`.
- **Ambiguos, sin mapear** (no adivinamos — revísalos contra el flujo/negocio):
  `Entrega y Otros Tramites` (24,114 — volumen alto, posible opción de menú no
  registrada en el maestro), `Solicitud_Para_mi`, `Transferencias script`,
  `Fuera de horario default`, `Finalización por fin de flujo`, más el resto de la
  cola larga (55 textos, ~106k pasos, no revisados uno a uno).

In [10]:
no_identificados = (
    df_join[df_join["matched_por"] == "sin_match"]
    .groupby("op_text")
    .size()
    .sort_values(ascending=False)
    .rename("n_ocurrencias")
    .reset_index()
)
print(f"{len(no_identificados)} textos distintos sin identificar ({no_identificados['n_ocurrencias'].sum()} pasos en total)")
no_identificados.head(40)

85 textos distintos sin identificar (852966 pasos en total)


,op_text,n_ocurrencias
0,Consulta ws GetClientByAni,82069
1,Usuario_Identificado_Con_ANI,82010
2,Consulta ws CheckAftersalesCases,46034
3,Numero documento ingresado,45699
4,Existe habeas data,35366
5,Consulta ws GetClientByDocument,34769
6,No input,31514
7,Usuario_Identificado_Con_Ani,31034
8,Primera pregunta SAC,29964
9,Bienvenida encuesta SAC,29964


#### Crosswalk manual de renombres

Solo incluimos pares que identificamos con confianza razonable comparando contra
el maestro y el flujo del PDF. Dos tipos:

1. **`CROSSWALK_EXACTO`**: el texto interno tiene un equivalente exacto en el
   maestro con otro nombre — se resuelve reintentando el cruce por texto
   normalizado usando el nombre del maestro, así se heredan `Nodo 1..10`,
   `Efectivo`, etc. completos, igual que un match real.
2. **`CROSSWALK_NODO_MANUAL`**: es navegación real pero no existe una fila
   equivalente en el maestro (sub-pasos de garantías no documentados a detalle).
   Se asigna un `nodo_final` aproximado a mano, marcado con
   `nodo_asignado_manual=True` para que quede trazable y auditable — no hereda
   `Efectivo`/`Clasifica Efectivo` porque no tenemos esa info real.

Los casos ambiguos (`Entrega y Otros Tramites`, `Solicitud_Para_mi`,
`Transferencias script`, `Fuera de horario default`) **no** se incluyen aquí a
propósito — mejor dejarlos como `no identificado` visible que adivinar mal.

In [11]:
CROSSWALK_EXACTO = {
    "Igual_O_Menor_30_Dias": "Periodo igual o menor a 30 dias",
    "Inicio IVR inbound transfers": "Inicio IVR",
}

CROSSWALK_NODO_MANUAL = {
    "Iniciar_Tu_Garantia": "Garantias y devoluciones",
    "Estado_De_Tu_Garantia": "Garantias y devoluciones",
}

# --- Aplicar CROSSWALK_EXACTO: reintenta el cruce por texto normalizado usando
# el nombre del maestro en vez del texto interno original ---
sin_match_actual = df_join["CodigoTraza_maestro"].isna()
mask_exacto = sin_match_actual & df_join["op_text"].isin(CROSSWALK_EXACTO)
if mask_exacto.any():
    texto_equivalente = df_join.loc[mask_exacto, "op_text"].map(CROSSWALK_EXACTO)
    for col in columnas_maestro_suffix:
        col_original = col.replace("_maestro", "")
        if col_original not in lookup_texto.columns:
            continue
        valores = texto_equivalente.apply(normalizar).map(lookup_texto[col_original])
        df_join.loc[mask_exacto, col] = df_join.loc[mask_exacto, col].fillna(valores)
    df_join.loc[mask_exacto, "matched_por"] = "crosswalk_exacto"

print(f"Recuperados por CROSSWALK_EXACTO: {mask_exacto.sum()} pasos")

# --- Aplicar CROSSWALK_NODO_MANUAL: asigna nodo_final directo, sin pasar por el maestro ---
df_join["nodo_asignado_manual"] = False
sin_match_actual = df_join["CodigoTraza_maestro"].isna()
mask_manual = sin_match_actual & df_join["op_text"].isin(CROSSWALK_NODO_MANUAL)
df_join.loc[mask_manual, "nodo_asignado_manual"] = True
df_join.loc[mask_manual, "matched_por"] = "crosswalk_nodo_manual"

print(f"Recuperados por CROSSWALK_NODO_MANUAL: {mask_manual.sum()} pasos")

# Recalcular sin_match definitivo después del crosswalk
df_join.loc[
    df_join["CodigoTraza_maestro"].isna() & ~df_join["op_text"].isin(CROSSWALK_NODO_MANUAL),
    "matched_por",
] = "sin_match"

print("\nmatched_por final:")
print(df_join["matched_por"].value_counts())
print(f"\n% total matcheado (incluye crosswalk): {(df_join['matched_por'] != 'sin_match').mean():.1%}")

Recuperados por CROSSWALK_EXACTO: 19490 pasos
Recuperados por CROSSWALK_NODO_MANUAL: 17226 pasos

matched_por final:
matched_por
sin_match                816250
llave                    427521
crosswalk_exacto          19490
crosswalk_nodo_manual     17226
texto_normalizado         15580
Name: count, dtype: int64

% total matcheado (incluye crosswalk): 37.0%


#### Categorización técnica / encuesta SAC

Independiente de si matcheó o no contra el maestro: marcamos qué pasos son
validación técnica de backend o la encuesta SAC, para poder excluirlos del grafo
de navegación más adelante sin perderlos de `df_transformado`.

In [12]:
TEXTOS_VALIDACION_TECNICA = [
    "Usuario_Identificado_Con_ANI",
    "Usuario_Identificado_Con_Ani",
    "Existe habeas data",
    "Numero documento ingresado",
    "Numero documento validado para factura",
    "Número de factura",
    "No input",
    "Repeat",
    "Pasa a buzon = NO",
]

patron_webservice = df_join["op_text"].str.startswith("Consulta ws", na=False)
patron_validacion = df_join["op_text"].isin(TEXTOS_VALIDACION_TECNICA)
patron_encuesta_sac = df_join["op_text"].str.contains("pregunta SAC|encuesta SAC", case=False, na=False, regex=True)

df_join["categoria_tecnica"] = "negocio"
df_join.loc[patron_webservice | patron_validacion, "categoria_tecnica"] = "validacion_tecnica"
df_join.loc[patron_encuesta_sac, "categoria_tecnica"] = "encuesta_sac"

print(df_join["categoria_tecnica"].value_counts())

categoria_tecnica
negocio               688029
validacion_tecnica    533776
encuesta_sac           74262
Name: count, dtype: int64


#### Revisión de lo que sigue sin identificar tras el crosswalk

Esto es lo que de verdad queda pendiente — ya sin lo recuperado por crosswalk ni
lo etiquetado como técnico/encuesta. Si `Entrega y Otros Tramites` sigue con
volumen alto, vale la pena registrarlo en el maestro (parece una opción de menú
real, no un tecnicismo).

In [13]:
pendientes = (
    df_join[(df_join["matched_por"] == "sin_match") & (df_join["categoria_tecnica"] == "negocio")]
    .groupby("op_text")
    .size()
    .sort_values(ascending=False)
    .rename("n_ocurrencias")
    .reset_index()
)
print(f"{len(pendientes)} textos de negocio genuinamente sin identificar ({pendientes['n_ocurrencias'].sum()} pasos)")
pendientes.head(20)

58 textos de negocio genuinamente sin identificar (208212 pasos)


,op_text,n_ocurrencias
0,Solicitud_Para_mi,26942
1,Entrega y Otros Tramites,24114
2,Paso agente SAC,20320
3,Finalización por fin de flujo,18541
4,Transferencias script,12010
5,Fuera de horario default,4989
6,Abandono_Cliente_Ivr,4818
7,Oficina Alkosto a SAC Despachos,4494
8,Transferencia_Exitosa,4349
9,Bot_Entiende_Si,4296


#### Clasificación final (`segmentacion`) y `nodo_final`

In [14]:
# Regla manual de respaldo: cualquier op_text que contenga "paso" se marca como
# Paso asesor — tiene prioridad sobre todo lo demás, igual que en 01_Alk_final.ipynb.
es_paso_asesor = df_join["op_text"].str.lower().str.contains("paso", na=False)

df_join["segmentacion"] = "no identificado"
df_join.loc[df_join["matched_por"] != "sin_match", "segmentacion"] = "Navegación"
df_join.loc[es_paso_asesor, "segmentacion"] = "Paso asesor"

print(df_join["segmentacion"].value_counts())
print("\nEfectivo (viene del maestro, Sí/No/nulo si no matcheó):")
print(df_join["Efectivo_maestro"].value_counts(dropna=False))

segmentacion
no identificado    792460
Navegación         450587
Paso asesor         53020
Name: count, dtype: int64

Efectivo (viene del maestro, Sí/No/nulo si no matcheó):
Efectivo_maestro
NaN    833476
No     447357
Si      15234
Name: count, dtype: int64


In [15]:
columnas_nodo_maestro = [f"Nodo {i}_maestro" for i in range(1, 11)]


def ultimo_nodo_no_nulo(fila):
    for col in reversed(columnas_nodo_maestro):
        valor = fila[col]
        if pd.notna(valor) and str(valor).strip().lower() not in ("", "null", "none"):
            return valor
    return None


df_join["nodo_final"] = df_join[columnas_nodo_maestro].apply(ultimo_nodo_no_nulo, axis=1)

# Los casos de CROSSWALK_NODO_MANUAL no tienen breadcrumb real del maestro —
# se les asigna directamente el nodo aproximado definido arriba.
mask_nodo_manual = df_join["nodo_asignado_manual"]
df_join.loc[mask_nodo_manual, "nodo_final"] = df_join.loc[mask_nodo_manual, "op_text"].map(CROSSWALK_NODO_MANUAL)

df_join["nodo_final"] = df_join["nodo_final"].fillna("no identificado")
df_join["nodo_final"].value_counts().head(15)

nodo_final
no identificado                               816250
Inicio IVR                                     95525
Menú Principal                                 78702
Garantias y devoluciones                       38901
¿Estado devolución en punto de venta? = NO     17090
Estado de entrega                              17041
¿Factura encontrada? = ERROR SI                15936
Habeas data positivo                           15900
Paso agente despachos                          14232
¿Total de facturas mayor a uno? = NO           13543
Informacion general                            13467
Validación factura correcta                    11694
Paso agente garantias                          10664
¿Factura igual a DES? = SI                     10338
¿Factura encontrada? = ERROR NO                 9891
Name: count, dtype: int64

#### Flag `incluir_en_grafo`

El grafo de navegación (Notebooks 4-5) se construye **solo** con los pasos donde
esto sea `True`: navegación de negocio o paso a asesor, excluyendo validación
técnica y encuesta SAC. `df_transformado` conserva todo, filtrado o no.

In [16]:
df_join["incluir_en_grafo"] = (
    df_join["segmentacion"].isin(["Navegación", "Paso asesor"])
    & (df_join["categoria_tecnica"] == "negocio")
)

print(df_join["incluir_en_grafo"].value_counts())
print(f"\n% de pasos que entrarán al grafo: {df_join['incluir_en_grafo'].mean():.1%}")

incluir_en_grafo
False    792460
True     503607
Name: count, dtype: int64

% de pasos que entrarán al grafo: 38.9%


#### Construcción de la arista origen→destino (`op_text_final`)

**Importante:** `op_text_final` se calcula sobre la secuencia **completa** de cada
conversación (no solo los pasos `incluir_en_grafo`), porque el orden real incluye
los pasos técnicos intercalados. El filtro por `incluir_en_grafo` se aplica
después, en el Notebook 5, al construir las aristas — si filtráramos antes de
hacer el `shift`, se generarían aristas falsas que saltan pasos intermedios reales.

In [19]:
df_join = df_join.sort_values(["id_conversacion", "orden"]).reset_index(drop=True)

df_join["op_text_final"] = df_join.groupby("id_conversacion")["op_text"].shift(-1)
ultimas_filas = df_join.groupby("id_conversacion").tail(1).index
df_join.loc[ultimas_filas, "op_text_final"] = "final"

df_join["orden"] = df_join.groupby("id_conversacion").cumcount()

df_join[["id_conversacion", "orden", "op_text", "op_text_final", "segmentacion", "nodo_final", "categoria_tecnica", "incluir_en_grafo"]].head(15)

,id_conversacion,orden,op_text,op_text_final,segmentacion,nodo_final,categoria_tecnica,incluir_en_grafo
0,0000b8f3-39e8-4388-aa08-23c0031a9d9f,0,Inicio IVR,Numero documento ingresado,Navegación,Inicio IVR,negocio,True
1,0000b8f3-39e8-4388-aa08-23c0031a9d9f,1,Numero documento ingresado,Habeas data negativo,no identificado,no identificado,validacion_tecnica,False
2,0000b8f3-39e8-4388-aa08-23c0031a9d9f,2,Habeas data negativo,Existe habeas data,Navegación,Habeas data negativo,negocio,True
3,0000b8f3-39e8-4388-aa08-23c0031a9d9f,3,Existe habeas data,Agendar servicio de instalación,no identificado,no identificado,validacion_tecnica,False
4,0000b8f3-39e8-4388-aa08-23c0031a9d9f,4,Agendar servicio de instalación,Usuario_Identificado_Con_ANI,Navegación,Agendar servicio de instalación,negocio,True
5,0000b8f3-39e8-4388-aa08-23c0031a9d9f,5,Usuario_Identificado_Con_ANI,Menu principal,no identificado,no identificado,validacion_tecnica,False
6,0000b8f3-39e8-4388-aa08-23c0031a9d9f,6,Menu principal,Consulta ws ConsultaHabeasData,Navegación,Menú Principal,negocio,True
7,0000b8f3-39e8-4388-aa08-23c0031a9d9f,7,Consulta ws ConsultaHabeasData,Consulta ws ActualizaHabeasData,no identificado,no identificado,validacion_tecnica,False
8,0000b8f3-39e8-4388-aa08-23c0031a9d9f,8,Consulta ws ActualizaHabeasData,Consulta ws CheckAftersalesCases,no identificado,no identificado,validacion_tecnica,False
9,0000b8f3-39e8-4388-aa08-23c0031a9d9f,9,Consulta ws CheckAftersalesCases,Consulta ws GetClientByAni,no identificado,no identificado,validacion_tecnica,False


#### Selección final de columnas y exportación

In [20]:
df_transformado = df_join.rename(
    columns={
        "Efectivo_maestro": "efectivo",
        "Clasifica Efectivo_maestro": "clasifica_efectivo",
        "CodigoTraza_maestro": "codigo_traza_maestro",
        "TrazaOpcion_maestro": "traza_opcion_maestro",
        "Nombre IVR_maestro": "nombre_ivr",
        "CodigoOrden_maestro": "codigo_orden",
    }
)[
    [
        "id_conversacion",
        "orden",
        "op_num",
        "op_text",
        "op_tiempo",
        "op_text_final",
        "origen_campo",
        "matched_por",
        "segmentacion",
        "categoria_tecnica",
        "incluir_en_grafo",
        "nodo_asignado_manual",
        "efectivo",
        "clasifica_efectivo",
        "nodo_final",
        "nombre_ivr",
        "codigo_orden",
    ]
    + columnas_nodo_maestro
]

df_transformado.to_parquet(OUT_TRANSFORMADO_PATH, index=False)
print(f"Guardado: {OUT_TRANSFORMADO_PATH}  ({len(df_transformado)} filas)")

muestra_path = PROC_DIR / f"df_transformado_muestra_{FECHA_INICIO}_{FECHA_FIN}.xlsx"
df_transformado.head(500).to_excel(muestra_path, index=False)
print(f"Muestra Excel: {muestra_path}")

Guardado: data\02_procesado\df_transformado_2026-06-01_2026-06-30.parquet  (1296067 filas)
Muestra Excel: data\02_procesado\df_transformado_muestra_2026-06-01_2026-06-30.xlsx


---
**Antes de seguir al Notebook 4:**
1. Revisa la tabla de `pendientes` — en especial `Entrega y Otros Tramites`
   (volumen alto). Si confirmas que es una opción de menú real, lo mejor es
   agregarla al maestro (`Detalle MaestroTrazasOpciones`) para la próxima corrida,
   en vez de seguir parchando con crosswalk manual acá.
2. Confirma que `% de pasos que entrarán al grafo` tiene sentido — debería rondar
   el volumen de `segmentacion` Navegación + Paso asesor menos lo técnico/encuesta.

**Siguiente paso:** `04_escenarios.ipynb` — usa la hoja `Ultima traza` del maestro
para identificar el escenario final de cada `id_conversacion` (equivalente al
viejo `traza_final`), filtra `df_transformado` por escenario y por cantidad de
pasos, y exporta los `base_*.xlsx` que consume el Notebook 5 (grafos) — aplicando
ahí el filtro `incluir_en_grafo` antes de construir las aristas.